# RBS Finance Dashboard – Colab Launcher
**執行順序：Cell 1 → Cell 2 → Cell 3（全部 runtime 重啟後只要重跑 Cell 3）**

| Cell | 功能 | Runtime 重啟後需要重跑？ |
|------|------|-------------------------|
| 1 | 掛載 Google Drive | 不需要 |
| 2 | 從 GitHub 同步最新程式碼 | 不需要（除非要更新版本） |
| 3 | 安裝套件 + 下載 cloudflared + 啟動 | **需要** |

In [ ]:
# ── CELL 1：掛載 Google Drive ────────────────────────────────────
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = Path('/content/drive/MyDrive/RBS_app')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

if str(DRIVE_BASE) not in sys.path:
    sys.path.insert(0, str(DRIVE_BASE))

print(f'Drive ready: {DRIVE_BASE}')

In [ ]:
# ── CELL 2：從 GitHub 同步最新 app.py / rbs_lib.py / stock_db.py ──
import urllib.request

REPO   = 'RoyARVSA/RBS_claude_finance'
BRANCH = 'claude/optimize-analysis-dashboard-NZUKB'

for fname in ['app.py', 'rbs_lib.py', 'stock_db.py']:
    # 用字串拼接避免 Markdown 把 URL 加上 <>
    url = 'https://raw.githubusercontent.com/' + REPO + '/' + BRANCH + '/' + fname
    dst = DRIVE_BASE / fname
    urllib.request.urlretrieve(url, dst)
    print(f'OK {fname} ({dst.stat().st_size/1024:.1f} KB)')

print('Sync done.')

In [ ]:
# ── CELL 3：安裝套件 + 啟動 Streamlit + cloudflared ─────────────
# Runtime 重啟後只需要重跑這個 cell
import subprocess, socket, time, sys, re, urllib.request
from pathlib import Path
from IPython.display import display, HTML

PORT     = 8501
APP_PATH = DRIVE_BASE / 'app.py'
LOG_PATH = DRIVE_BASE / 'streamlit.log'
CF_BIN   = '/tmp/cloudflared'   # /tmp 每次 runtime 都清空，在這裡重新下載
CF_LOG   = '/tmp/cloudflared.log'

# ── 1. 安裝 Python 套件 ──────────────────────────────────────────
print('Installing packages...')
subprocess.run(
    [sys.executable, '-m', 'pip', '-q', 'install',
     'streamlit', 'yfinance', 'numpy', 'pandas',
     'matplotlib', 'seaborn', 'scikit-learn', 'scipy',
     'adjustText', 'feedparser', 'requests', 'plotly', 'openai'],
    check=True
)
print('Packages ready.')

# ── 2. 下載 cloudflared ──────────────────────────────────────────
cf_path = Path(CF_BIN)
if not cf_path.exists() or cf_path.stat().st_size < 1_000_000:
    print('Downloading cloudflared...')
    url = ('https://github.com/cloudflare/cloudflared'
           '/releases/latest/download/cloudflared-linux-amd64')
    urllib.request.urlretrieve(url, CF_BIN)
    cf_path.chmod(0o755)
    print(f'cloudflared ready ({cf_path.stat().st_size/1024/1024:.1f} MB)')
else:
    print('cloudflared already exists.')

# ── 3. 清除舊進程 ────────────────────────────────────────────────
subprocess.run(['fuser', '-k', f'{PORT}/tcp'], capture_output=True)
time.sleep(0.5)

# ── 4. 啟動 Streamlit ────────────────────────────────────────────
print(f'Starting Streamlit on port {PORT}...')
_log = open(LOG_PATH, 'w')
st_proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', str(APP_PATH),
     '--server.port',                str(PORT),
     '--server.address',             '0.0.0.0',
     '--server.headless',            'true',
     '--server.enableCORS',          'false',
     '--server.enableXsrfProtection','false',
     '--browser.gatherUsageStats',   'false'],
    stdout=_log, stderr=_log,
)

# 等待就緒
t0 = time.time()
while time.time() - t0 < 60:
    try:
        socket.create_connection(('127.0.0.1', PORT), 1).close()
        print(f'Streamlit ready ({time.time()-t0:.1f}s)')
        break
    except OSError:
        time.sleep(0.3)
else:
    print('--- Streamlit log ---')
    print(Path(LOG_PATH).read_text()[-1000:])
    raise RuntimeError('Streamlit failed to start')

# ── 5. 啟動 cloudflared tunnel ───────────────────────────────────
print('Starting cloudflared tunnel...')
cf_proc = subprocess.Popen(
    [CF_BIN, 'tunnel', '--url', 'http://localhost:' + str(PORT)],
    stdout=open(CF_LOG, 'w'), stderr=subprocess.STDOUT,
)

# 等待 URL 出現
pub = None
for _ in range(60):
    time.sleep(0.5)
    match = re.search(
        r'https://[a-z0-9\-]+\.trycloudflare\.com',
        open(CF_LOG).read()
    )
    if match:
        pub = match.group(0)
        break

if not pub:
    print('--- cloudflared log ---')
    print(open(CF_LOG).read())
    raise RuntimeError('cloudflared tunnel failed')

# ── 6. 顯示結果 ──────────────────────────────────────────────────
print(f'\nPublic URL: {pub}')
print(f'Log:        {LOG_PATH}')
print('To stop:    st_proc.terminate(); cf_proc.terminate()')
display(HTML(f'<a href="{pub}" target="_blank" style="font-size:18px">👉 點此開啟 Dashboard</a>'))